In [37]:
# imports
import pandas as pd
import requests
import yfinance as yf
import numpy as np
from transformers import pipeline
from datetime import datetime, timedelta
import finnhub
from dotenv import load_dotenv
import os
from edgar import Company, set_identity


### Gathering Data

In [2]:
headers = {"User-Agent": "Mozilla/5.0 (compatible; my-script/1.0)"}
url = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'
html = requests.get(url, headers=headers).text
table = pd.read_html(html)[0]
table['Symbol'] = table['Symbol'].str.replace('.', '-').tolist()
table.head()

/var/folders/5l/vsm1wvjn4qdg25qwjfcj83kr0000gn/T/ipykernel_67548/4004594965.py:4: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  table = pd.read_html(html)[0]


,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,Date added,CIK,Founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989


#### Tickers
**Semiconductors**: NVDA (NVIDIA), AMD (Advanced Micro Devices), INTC (Intel), ASML (ASML Holding), TSM (Taiwan Semiconductor Manufacturing), AVGO (Broadcom), 
    MRVL (Marvell Technology), ARM (Arm Holdings), SMCI (Super Micro Computer)<br>
**Hyperscalars**: MSFT (Microsoft), GOOGL (Google), AMZN (Amazon), META (Facebook) <br>
**AI Pure Plays**: PLTR (Palantir), AI (C3.ai), SOUN (Soundhound AI) <br>
**Infrastructure**: DELL (Dell Technologies), ANET (Arista Networks), HPE (Hewlett Packard Enterprise)

In [10]:
# list of all the tickers with a dictionary that sorts them by type of stock
tickers = pd.read_csv('../../data/tickers.csv')
tickers = tickers.drop(columns=['Unnamed: 0'])

ai_tickers = {
    'semis':   ['NVDA','AMD','INTC','ASML','TSM','AVGO','MRVL','ARM','SMCI'], # semiconductors
    'hypers':  ['MSFT','GOOGL','AMZN','META'], # hyperscalar
    'pure':    ['PLTR','AI','SOUN'], # AI pure plays
    'infra':   ['DELL','ANET','HPE'] # infrastructure
}
all_tickers = [t for group in ai_tickers.values() for t in group]

In [13]:
# Data from 2018 - 2026 (pre-COVID to present)
prices = yf.download(
    tickers=all_tickers + ['SOXX', 'QQQ'],
    start='2018-01-01',
    end='2026-01-01',
    auto_adjust=True,
    progress=False
)
prices.to_csv('../../data/prices_raw.csv')

In [14]:
prices.head()

Price      Close                                                          \
Ticker        AI    AMD       AMZN       ANET ARM        ASML       AVGO   
Date                                                                       
2018-01-02   NaN  10.98  59.450500  14.439375 NaN  163.685883  21.053345   
2018-01-03   NaN  11.55  60.209999  14.725000 NaN  164.929214  21.283579   
2018-01-04   NaN  12.12  60.479500  14.543125 NaN  166.467270  21.290672   
2018-01-05   NaN  11.88  61.457001  14.798125 NaN  168.419754  21.416830   
2018-01-08   NaN  12.28  62.343498  15.691250 NaN  169.303879  21.468082   

Price                                        ...    Volume                     \
Ticker           DELL      GOOGL        HPE  ...      META     MRVL      MSFT   
Date                                         ...                                
2018-01-02  21.049423  53.188866  11.300008  ...  18151900  5619900  22483800   
2018-01-03  21.271585  54.096317  11.369617  ...  16886600  9014000  26061400   
2018-01-04  21.649519  54.306450  11.648057  ...  13880900  9221200  21912000   
2018-01-05  21.807842  55.026566  11.640326  ...  13574500  5855400  23407100   
2018-01-08  21.802734  55.220848  11.462432  ...  17994700  5529500  22113000   

Price                                                                 
Ticker           NVDA PLTR       QQQ     SMCI SOUN     SOXX      TSM  
Date                                                                  
2018-01-02  355616000  NaN  32573300  4344000  NaN  2860200  4984000  
2018-01-03  914704000  NaN  29383600  4094000  NaN  1326900  6963200  
2018-01-04  583268000  NaN  24776100  3631000  NaN  1335000  4876600  
2018-01-05  580124000  NaN  26992300  2178000  NaN  1170600  5330800  
2018-01-08  881216000  NaN  23159100  3484000  NaN  1622700  3538200  

[5 rows x 105 columns]

### Cleaning Data

In [15]:
close  = prices['Close']
volume = prices['Volume']
soxx   = close['SOXX']
qqq    = close['QQQ']

In [17]:
# Remove tickers with more than 5% NaN values
clean_prices = close.loc[:, close.isna().mean() <= 0.05]

# Forward fill short gaps of 1-2 days
clean_prices = clean_prices.ffill(limit=2)
clean_prices = clean_prices.dropna()
clean_prices.to_csv('../../data/clean_prices.csv')

In [18]:
# Convert prices to log
log_ret = np.log(clean_prices / clean_prices.shift(1)).dropna()

### Feature Engineering

In [19]:
# SOXX: holds 30 largest semiconductor companies
# QQQ: holds 100 largest non-financial companies on the NASDAQ, heavily weighted towards big tech
def make_features(close: pd.Series,
                     volume: pd.Series,
                     soxx: pd.Series,
                     qqq: pd.Series) -> pd.DataFrame:
    df = pd.DataFrame(index=close.index)

    # Lag returns: percent change over x days
    for lag in [1, 3, 5, 10, 20]:
        df[f'ret_{lag}d'] = close.pct_change(lag)

    # Rolling volatility
    df['vol_5d']  = close.pct_change().rolling(5).std()
    df['vol_20d'] = close.pct_change().rolling(20).std()

    # RSI (14-day): Relative Strength Index; measures magnitude and recent price changes in stocks
    delta = close.pct_change()
    gain  = delta.clip(lower=0).rolling(14).mean()
    loss  = (-delta.clip(upper=0)).rolling(14).mean()
    df['rsi_14'] = 100 - (100 / (1 + gain / loss))

    # How does the stock compare to SOXX and QQQ?
    df['rel_soxx'] = close.pct_change(20) - soxx.pct_change(20)
    df['rel_qqq']  = close.pct_change(20) - qqq.pct_change(20)

    # Volume spike: number of shares traded on a given day
    df['vol_spike'] = volume / volume.rolling(20).mean()

    # 52-week high proximity: the highest price traded at over the last 252 trading days (1 year)
    df['pct_from_52wk_high'] = close / close.rolling(252).max() - 1

    return df.dropna()

In [20]:
all_feature_dfs = []
for ticker in all_tickers:
    if ticker not in close.columns:
        continue
    ticker_feats = make_features(
        close  = close[ticker],
        volume = volume[ticker],
        soxx   = soxx,
        qqq    = qqq
    )
    ticker_feats['Ticker'] = ticker      
    all_feature_dfs.append(ticker_feats)

In [21]:
features_df = pd.concat(all_feature_dfs).reset_index()
features_df = features_df.set_index(['Date','Ticker'])

In [22]:
# Did this stock go up over the next 5 days?
features_df['Target'] = (
    features_df.groupby('Ticker')['ret_1d']
    .transform(lambda x: x.shift(-1).rolling(5).sum() > 0)
).astype(int)

In [23]:
features_df.head()

,,ret_1d,ret_3d,ret_5d,ret_10d,ret_20d,vol_5d,vol_20d,rsi_14,rel_soxx,rel_qqq,vol_spike,pct_from_52wk_high,Target
Date,Ticker,,,,,,,,,,,,,
2019-01-02,NVDA,0.020374,0.038499,0.071923,-0.051261,-0.166493,0.023554,0.034328,37.207807,-0.107087,-0.083557,0.785101,-0.528766,0
2019-01-03,NVDA,-0.060417,-0.042349,-0.038392,-0.128964,-0.247295,0.033030,0.034168,30.211013,-0.110360,-0.119161,1.104590,-0.557236,0
2019-01-04,NVDA,0.064068,0.020150,0.038271,-0.016750,-0.133155,0.045219,0.035031,40.655556,-0.080480,-0.078587,0.933377,-0.528870,0
2019-01-07,NVDA,0.052940,0.052709,0.072952,0.061436,-0.094068,0.049557,0.037384,48.847603,-0.063841,-0.044049,1.128762,-0.503928,0
2019-01-08,NVDA,-0.024896,0.092507,0.047415,0.079185,-0.052707,0.052560,0.034712,48.297088,-0.055655,-0.044009,1.240790,-0.516278,1


In [24]:
features_df.to_csv('../../data/features_df.csv')

### Continuous News Sentiment with FinnHub

In [26]:
load_dotenv()

True

In [27]:
client = finnhub.Client(api_key=os.getenv("FINNHUB_API_KEY"))

In [ ]:
# date, headline, and summary of most recent 50 news per ticker
def get_ai_news(ticker, start_date, end_date):
    start_ts = int(datetime.strptime(start_date, "%Y-%m-%d").timestamp())
    end_ts   = int(datetime.strptime(end_date,   "%Y-%m-%d").timestamp())

    news = client.company_news(ticker, _from=start_date, to=end_date)

    articles = []
    for item in news[:500]:
        articles.append({
            'date':     datetime.fromtimestamp(item['datetime']).date(),
            'headline': item['headline'],
            'summary':  item.get('summary', '')
        })
    return pd.DataFrame(articles)

In [29]:
finbert = pipeline('sentiment-analysis', model='ProsusAI/finbert', truncation=True, max_length=512)

Device set to use mps:0


In [30]:
def score_headlines(df):
    texts = (df['headline'] + '. ' + df['summary']).tolist()
    results = finbert(texts)
    df['sentiment'] = [r['label'].lower() for r in results]
    df['score'] = [r['score'] if r['label'] == 'positive' else -r['score'] if r['label'] == 'negative' else 0 for r in results]
    daily = df.groupby('date').agg(
        news_sentiment = ('score', 'mean'),
        news_count = ('score', 'count'),
        pct_positive = ('sentiment', lambda x: (x=='positive').mean())
    )
    return daily

In [31]:
def get_sentiment_for_ticker(ticker, df_ticker):
    # find the beginning and end dates for each ticker
    start_date = df_ticker.index.get_level_values('Date').min().strftime("%Y-%m-%d")
    end_date   = df_ticker.index.get_level_values('Date').max().strftime("%Y-%m-%d")
    
    news_df = get_ai_news(ticker, start_date, end_date)
    if news_df.empty:
        return pd.DataFrame()
    
    # provides sentiment rating based on 50 most recent news for each ticker
    daily_sentiment = score_headlines(news_df)
    daily_sentiment['Ticker'] = ticker
    return daily_sentiment


In [33]:
# Loop over all tickers
all_sentiment = []
for ticker in features_df.index.get_level_values('Ticker').unique():
    df_ticker = features_df.xs(ticker, level='Ticker')
    sentiment = get_sentiment_for_ticker(ticker, df_ticker)
    if not sentiment.empty:
        all_sentiment.append(sentiment)

sentiment_df = pd.concat(all_sentiment)
sentiment_df.index = pd.to_datetime(sentiment_df.index)
sentiment_df.to_csv('../../data/sentiment_df.csv')

In [34]:
# merge sentiment df back to features df
df = features_df.reset_index()
df['Date'] = pd.to_datetime(df['Date'])
df = df.merge(
    sentiment_df.reset_index().rename(columns={'date': 'Date'}),
    on=['Date', 'Ticker'],
    how='left'
)
df = df.set_index(['Date', 'Ticker'])

# Fill missing sentiment (days with no news)
df[['news_sentiment', 'new_count', 'pct_positive']] = df[['news_sentiment', 'news_count', 'pct_positive']].fillna(0)

In [35]:
# Domain keyword dictionaries
GPU_DEMAND_KEYWORDS = [
    # current & next-gen hardware
    'h100', 'h200', 'h20',
    'blackwell', 'b100', 'b200', 'gb200', 'gb300',
    'rubin', 'vera rubin', 'nvl72',
    'dgx', 'hgx',
    # demand signals
    'gpu demand', 'gpu supply', 'gpu allocation',
    'data center revenue', 'data center demand',
    'inference demand', 'inference workload',
    'training compute', 'compute demand',
    # software/platform
    'cuda', 'tensorrt', 'triton', 'nim microservices',
    'hopper architecture', 'blackwell architecture'
    # other GPU demand words
    'cloud ai', 'ai infrastructure', 'ai accelerator', 'tensor processing', 'custom ai chip'
]

CAPEX_UP_KEYWORDS = [
    'increasing capex', 'raised capex', 'accelerating capex',
    'accelerating investment', 'capital expenditure increase',
    'expanding capacity', 'record investment',
    'investing heavily', 'multiyear investment',
    'committed to investing', 'significant investment in ai',
    'capital investment', 'infrastructure investment', 'ai investment'
]

CAPEX_DOWN_KEYWORDS = [
    'reducing capex', 'cutting investment', 'slowing spend',
    'deferring investment', 'pausing investment',
    'capital efficiency', 'disciplined spending',
    'revisiting capex', 'moderating investment'
]

COMPETITIVE_RISK_KEYWORDS = [
    # rival chips by name
    'amd instinct', 'gaudi', 'trainium', 'inferentia', 'maia',
    'google tpu', 'axion',
    # strategic alternatives
    'custom silicon', 'in-house chip', 'in-house ai chip',
    'merchant silicon', 'proprietary chip',
    'competitive alternative', 'alternative to nvidia'
]

In [36]:
def extract_domain_signals(transcript_text):
    text_lower = transcript_text.lower()
    signals = {}

    # Raw mention counts (normalize by transcript length)
    word_count = len(text_lower.split())
    signals['gpu_mentions']    = sum(text_lower.count(k) for k in GPU_DEMAND_KEYWORDS) / word_count * 1000
    signals['capex_up_score']  = sum(text_lower.count(k) for k in CAPEX_UP_KEYWORDS) / word_count * 1000
    signals['capex_down_score']= sum(text_lower.count(k) for k in CAPEX_DOWN_KEYWORDS) / word_count * 1000
    competitor_keys = [k for k in COMPETITIVE_RISK_KEYWORDS if k != ticker.lower()]
    signals['competitor_mentions'] = sum(text_lower.count(k) for k in competitor_keys) / word_count * 1000
    signals['capex_net']       = signals['capex_up_score'] - signals['capex_down_score'] 

    # Paragraph-level: find sentences mentioning AI demand
    ai_sentences = [s for s in transcript_text.split('.')
                    if any(k in s.lower() for k in ['artificial intelligence','ai demand','ai revenue'])]
    signals['ai_sentence_ratio'] = len(ai_sentences) / max(len(transcript_text.split('.')),1)

    return signals

In [ ]:
set_identity(os.getenv('EMAIL'))

TypeError: str expected, not NoneType